# COVID-19 Informative Tweet Risk Classification
## TF-IDF + LinearSVC 개선 모델

이 노트북은 기존 `TF-IDF + Logistic Regression` baseline을 개선한 버전입니다.

핵심 변경점:
- `LogisticRegression` 대신 `LinearSVC` 사용
- word n-gram과 character n-gram을 함께 사용
- class imbalance 대응을 위해 `class_weight="balanced"` 적용
- train/valid/test split을 분리해 test leakage 방지
- 최종 test set은 마지막에 한 번만 평가

## 1. Google Drive mount and package import

In [1]:
# Colab에서 Google Drive를 사용하는 경우 실행하세요.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import os
import glob
import random

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Path and configuration

`DATA_PATH`는 본인 Google Drive 경로에 맞게 수정하세요.
만약 Colab 왼쪽 파일 탭에 TSV를 직접 업로드했다면 `/content/*.tsv`에서 자동으로 찾도록 했습니다.

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/AI@Sogang_4_M2"
DATA_PATH = f"{PROJECT_DIR}/data/raw/covid_informative_risk_dataset_all.tsv"
SAVE_DIR = f"{PROJECT_DIR}/models/risk_tfidf_linearsvc"
os.makedirs(SAVE_DIR, exist_ok=True)

# Drive 경로에 없으면 Colab 세션에 직접 업로드한 TSV 파일을 자동 탐색합니다.
if not os.path.exists(DATA_PATH):
    candidate_paths = glob.glob("/content/*risk*.tsv") + glob.glob("/content/*.tsv")
    if candidate_paths:
        DATA_PATH = candidate_paths[0]
    else:
        raise FileNotFoundError(
            "TSV 파일을 찾지 못했습니다. DATA_PATH를 수정하거나 Colab에 TSV를 업로드하세요."
        )

TEXT_COL = "Text"
LABEL_COL = "Risk_Level"

LABEL2ID = {"Low": 0, "Medium": 1, "High": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABEL_NAMES = [ID2LABEL[i] for i in range(len(ID2LABEL))]

print("DATA_PATH:", DATA_PATH)
print("SAVE_DIR:", SAVE_DIR)

DATA_PATH: /content/drive/MyDrive/AI@Sogang_4_M2/data/raw/covid_informative_risk_dataset_all.tsv
SAVE_DIR: /content/drive/MyDrive/AI@Sogang_4_M2/models/risk_tfidf_linearsvc


## 3. Load and clean dataset

In [4]:
df = pd.read_csv(DATA_PATH, sep="\t")
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

df[TEXT_COL] = df[TEXT_COL].astype("string").str.strip()
df = df[df[TEXT_COL].notna()]
df = df[df[TEXT_COL] != ""]

df[LABEL_COL] = df[LABEL_COL].astype("string").str.strip()
df = df[df[LABEL_COL].isin(LABEL2ID.keys())]

# 같은 tweet text가 중복될 경우 leakage를 막기 위해 먼저 제거합니다.
df = df.drop_duplicates(subset=[TEXT_COL], keep="first")
df = df[[TEXT_COL, LABEL_COL]].copy()
df["label_id"] = df[LABEL_COL].map(LABEL2ID).astype(int)

print("Clean shape:", df.shape)
print("Label counts:")
print(df[LABEL_COL].value_counts())
print("\nLabel ratio:")
print((df[LABEL_COL].value_counts(normalize=True) * 100).round(2))

Raw shape: (4689, 6)
Columns: ['Id', 'Text', 'Info_Label', 'Risk_Level', 'Risk_Label_Reason', 'Original_Split']
Clean shape: (4685, 3)
Label counts:
Risk_Level
High      2681
Medium    1181
Low        823
Name: count, dtype: Int64

Label ratio:
Risk_Level
High      57.23
Medium    25.21
Low       17.57
Name: proportion, dtype: Float64


## 4. Stratified train / valid / test split

기존 노트북은 test set으로 매 epoch/model selection을 하는 구조라서 test leakage 위험이 있었습니다.
여기서는 train 70%, valid 10%, test 20%로 나눕니다.

In [5]:
train_valid_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["label_id"],
)

train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=0.125,  # 0.8 * 0.125 = 0.1, so final valid ratio is 10%.
    random_state=RANDOM_STATE,
    stratify=train_valid_df["label_id"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(f"{name}: {len(split_df)}")
    print(split_df[LABEL_COL].value_counts().reindex(LABEL2ID.keys()))
    print()

train: 3279
Risk_Level
Low        576
Medium     827
High      1876
Name: count, dtype: Int64

valid: 469
Risk_Level
Low        82
Medium    118
High      269
Name: count, dtype: Int64

test: 937
Risk_Level
Low       165
Medium    236
High      536
Name: count, dtype: Int64



## 5. Build TF-IDF + LinearSVC candidates

검증용으로 몇 가지 후보를 비교합니다. 최종 모델 선택은 valid macro F1 기준입니다.

In [6]:
def build_word_svc(c=1.0):
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                LinearSVC(
                    C=c,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def build_word_char_svc(c=0.5):
    return Pipeline(
        steps=[
            (
                "features",
                FeatureUnion(
                    [
                        (
                            "word",
                            TfidfVectorizer(
                                lowercase=True,
                                analyzer="word",
                                ngram_range=(1, 2),
                                min_df=2,
                                max_features=25000,
                                sublinear_tf=True,
                            ),
                        ),
                        (
                            "char",
                            TfidfVectorizer(
                                lowercase=True,
                                analyzer="char_wb",
                                ngram_range=(3, 5),
                                min_df=3,
                                max_features=25000,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            (
                "classifier",
                LinearSVC(
                    C=c,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

candidates = {
    "word_1_2_svc_C1.0": build_word_svc(c=1.0),
    "word_char_svc_C1.0": build_word_char_svc(c=1.0),
    "word_char_svc_C0.5": build_word_char_svc(c=0.5),
}

results = []

for model_name, candidate in candidates.items():
    print("=" * 80)
    print("Training:", model_name)
    candidate.fit(train_df[TEXT_COL], train_df["label_id"])
    valid_pred = candidate.predict(valid_df[TEXT_COL])
    valid_acc = accuracy_score(valid_df["label_id"], valid_pred)
    valid_macro_f1 = f1_score(valid_df["label_id"], valid_pred, average="macro")
    results.append({
        "model_name": model_name,
        "valid_accuracy": valid_acc,
        "valid_macro_f1": valid_macro_f1,
    })
    print(f"Valid Accuracy: {valid_acc:.4f}")
    print(f"Valid Macro F1: {valid_macro_f1:.4f}")

results_df = pd.DataFrame(results).sort_values("valid_macro_f1", ascending=False)
results_df

Training: word_1_2_svc_C1.0
Valid Accuracy: 0.8209
Valid Macro F1: 0.8025
Training: word_char_svc_C1.0
Valid Accuracy: 0.8380
Valid Macro F1: 0.8230
Training: word_char_svc_C0.5
Valid Accuracy: 0.8401
Valid Macro F1: 0.8261


,model_name,valid_accuracy,valid_macro_f1
2,word_char_svc_C0.5,0.840085,0.826128
1,word_char_svc_C1.0,0.837953,0.822972
0,word_1_2_svc_C1.0,0.820896,0.802536


## 6. Retrain best model on train+valid, then final test evaluation

In [7]:
best_model_name = results_df.iloc[0]["model_name"]
print("Best model by valid macro F1:", best_model_name)

best_model = clone(candidates[best_model_name])
final_train_df = pd.concat([train_df, valid_df], axis=0).reset_index(drop=True)

best_model.fit(final_train_df[TEXT_COL], final_train_df["label_id"])
test_pred = best_model.predict(test_df[TEXT_COL])

test_accuracy = accuracy_score(test_df["label_id"], test_pred)
test_macro_f1 = f1_score(test_df["label_id"], test_pred, average="macro")

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    test_df["label_id"],
    test_pred,
    labels=[0, 1, 2],
    target_names=LABEL_NAMES,
    zero_division=0,
))
print("Confusion matrix [Low, Medium, High]:")
print(confusion_matrix(test_df["label_id"], test_pred, labels=[0, 1, 2]))

Best model by valid macro F1: word_char_svc_C0.5
Test Accuracy: 0.8356
Test Macro F1: 0.8158

Classification report:
              precision    recall  f1-score   support

         Low       0.91      0.77      0.83       165
      Medium       0.73      0.74      0.73       236
        High       0.87      0.90      0.88       536

    accuracy                           0.84       937
   macro avg       0.83      0.80      0.82       937
weighted avg       0.84      0.84      0.84       937

Confusion matrix [Low, Medium, High]:
[[127  16  22]
 [  8 175  53]
 [  5  50 481]]


## 7. Error analysis helper

In [8]:
analysis_df = test_df.copy()
analysis_df["pred_id"] = test_pred
analysis_df["Predicted_Risk_Level"] = analysis_df["pred_id"].map(ID2LABEL)
wrong_df = analysis_df[analysis_df["label_id"] != analysis_df["pred_id"]].copy()

print("Wrong predictions:", len(wrong_df), "/", len(test_df))
wrong_df[[TEXT_COL, LABEL_COL, "Predicted_Risk_Level"]].head(20)

Wrong predictions: 154 / 937


,Text,Risk_Level,Predicted_Risk_Level
3,"In Ballad’s news conference Thursday, Presiden...",Medium,High
11,A Carnegie Mellon University back from spring ...,High,Medium
18,NEWS: The health minister Nadine Dorries has b...,Low,Medium
20,COVID-19 update: PA’s testing capabilities are...,High,Medium
28,Some positive news from Niagara County: the fi...,Low,High
37,TONIGHT: @USER speaks with Christ Church Georg...,Low,High
40,"In #Brazil, Fear Surges Along with #Coronaviru...",Medium,High
51,My God: #America lost 4591 souls in one day fr...,Medium,High
60,Second person dies of coronavirus in UK says H...,Low,High
66,US has potential to become the new coronavirus...,Low,Medium


## 8. Save model and config

In [9]:
MODEL_SAVE_PATH = f"{SAVE_DIR}/risk_tfidf_linearsvc.joblib"
CONFIG_SAVE_PATH = f"{SAVE_DIR}/config.json"

joblib.dump(best_model, MODEL_SAVE_PATH)

config = {
    "model_type": "tfidf_linearsvc",
    "best_model_name": best_model_name,
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()},
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "random_state": RANDOM_STATE,
    "split": {"train": 0.7, "valid": 0.1, "test": 0.2},
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
}

with open(CONFIG_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved model:", MODEL_SAVE_PATH)
print("Saved config:", CONFIG_SAVE_PATH)

Saved model: /content/drive/MyDrive/AI@Sogang_4_M2/models/risk_tfidf_linearsvc/risk_tfidf_linearsvc.joblib
Saved config: /content/drive/MyDrive/AI@Sogang_4_M2/models/risk_tfidf_linearsvc/config.json


## 9. Single tweet prediction

`LinearSVC`는 `predict_proba()`를 기본 제공하지 않으므로 확률 대신 decision score를 확인합니다.

In [10]:
def predict_tweet(text: str):
    pred_id = int(best_model.predict([text])[0])
    decision_scores = best_model.decision_function([text])[0]
    score_dict = {ID2LABEL[i]: float(decision_scores[i]) for i in range(len(ID2LABEL))}
    return {
        "text": text,
        "predicted_label": ID2LABEL[pred_id],
        "decision_scores": score_dict,
    }

predict_tweet("New COVID-19 deaths and cases were reported today in the city.")

{'text': 'New COVID-19 deaths and cases were reported today in the city.',
 'predicted_label': 'High',
 'decision_scores': {'Low': -0.8030805561814917,
  'Medium': -2.6750586992087304,
  'High': 1.8118679367900796}}